In [0]:
%python
from pyspark.dbutils import *
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import builtins
spark=SparkSession.builder.appName("Spark DataFrames").getOrCreate()

In [0]:
%skip
%sql

CREATE TABLE IF NOT EXISTS workspace.default.upi_pipeline_metrics (
    run_id STRING,
    run_date TIMESTAMP,
    txn_bronze_count LONG,
    settlement_bronze_count LONG,
    settlement_rescued_count LONG,
    txn_silver_count LONG,
    txn_silver_reject_count LONG,
    settlement_silver_count LONG,
    txn_rejection_rate DOUBLE,
    gold_count LONG,
    gold_discrep_unsettled_success_count LONG,
    gold_discrep_stale_pending_count LONG,
    orphan_record_count LONG,
    discrepancy_rate DOUBLE,
    bronze_duration DOUBLE,
    silver_duration DOUBLE,
    gold_duration DOUBLE,
    total_duration DOUBLE
    
)

In [0]:
%python
pipeline_id = dbutils.widgets.get("pipeline_id")
run_date = dbutils.widgets.get("run_date")

print(pipeline_id)
print(run_date)

In [0]:
%python
event_log_df = spark.sql(f"SELECT * FROM event_log('{pipeline_id}')")

run_id = event_log_df.filter(col("event_type") == "create_update")\
.orderBy(col("timestamp").desc()).select("origin.update_id").limit(1).collect()[0]["update_id"]

event_log_df = event_log_df.filter(col("event_type")=="flow_progress").filter(col("origin.update_id") == lit(run_id))

In [0]:
%python

txn_bronze_count = event_log_df.filter(col("origin.flow_name")=="workspace.default.upi_transactions_bronze")\
.agg(sum(expr("details:flow_progress:metrics:num_output_rows").cast("int")).alias("txn_bronze_count")).collect()[0]["txn_bronze_count"]

if txn_bronze_count is None:
    txn_bronze_count =0

data =[run_id,run_date,txn_bronze_count]
cols=["run_id","run_date","txn_bronze_count"]
txn_br_count_df = spark.createDataFrame([data],cols)

txn_br_count_df.createOrReplaceTempView("txn_br_count_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING txn_br_count_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.txn_bronze_count = s.txn_bronze_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, txn_bronze_count) VALUES (s.run_id,s.run_date,s.txn_bronze_count)
""")

In [0]:
settlement_bronze_count = event_log_df.filter(col("origin.flow_name")=="workspace.default.bronze_settlement_data")\
.agg(sum(expr("details:flow_progress:metrics:num_output_rows").cast("int")).alias("settlement_bronze_count")).collect()[0]["settlement_bronze_count"]

if settlement_bronze_count is None:
    settlement_bronze_count =0

data =[run_id,run_date,settlement_bronze_count]
cols=["run_id","run_date","settlement_bronze_count"]
txn_br_count_df = spark.createDataFrame([data],cols)

txn_br_count_df.createOrReplaceTempView("settlement_bronze_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING settlement_bronze_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.settlement_bronze_count = s.settlement_bronze_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, settlement_bronze_count) VALUES (s.run_id,s.run_date,s.settlement_bronze_count)
""")

In [0]:
time_boundaries = event_log_df.filter(col("origin.flow_name")=="workspace.default.bronze_settlement_data")

boundaries = time_boundaries.agg(min("timestamp").cast("string").alias("run_start"),max("timestamp").cast("string").alias("run_end"))\
    .collect()[0]

run_start = boundaries["run_start"]
run_end = boundaries["run_end"]



In [0]:
settlement_rescued_count = spark.read.table("workspace.default.bronze_settlement_data").filter(col("_rescued_data").isNotNull())\
.filter((col("ingested_time")<to_timestamp(lit(run_end))) & (col("ingested_time")>to_timestamp(lit(run_start)))).count()

if settlement_rescued_count is None:
    settlement_rescued_count =0

data =[run_id,run_date,settlement_rescued_count]
cols=["run_id","run_date","settlement_rescued_count"]
settlement_rescued_count_df = spark.createDataFrame([data],cols)

settlement_rescued_count_df.createOrReplaceTempView("settlement_rescued_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING settlement_rescued_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.settlement_rescued_count = s.settlement_rescued_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, settlement_rescued_count) VALUES (s.run_id,s.run_date,s.settlement_rescued_count)
""")

In [0]:
txn_silver_count = event_log_df.filter(col("origin.flow_name")=="workspace.default.upi_transactions_silver")\
.agg(sum(expr("details:flow_progress:metrics:num_output_rows").cast("int")).alias("txn_silver_count")).collect()[0]["txn_silver_count"]

if txn_silver_count is None:
    txn_silver_count =0

data =[run_id,run_date,txn_silver_count]
cols=["run_id","run_date","txn_silver_count"]
txn_silver_count_df = spark.createDataFrame([data],cols)

txn_silver_count_df.createOrReplaceTempView("txn_silver_count_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING txn_silver_count_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.txn_silver_count = s.txn_silver_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, txn_silver_count) VALUES (s.run_id,s.run_date,s.txn_silver_count)
""")

In [0]:
txn_silver_reject_count = event_log_df.filter(col("origin.flow_name")=="workspace.default.upi_transactions_quarantine")\
.agg(sum(expr("details:flow_progress:metrics:num_output_rows").cast("int")).alias("txn_silver_reject_count")).collect()[0]["txn_silver_reject_count"]

if txn_silver_reject_count is None:
    txn_silver_reject_count =0

data =[run_id,run_date,txn_silver_reject_count]
cols=["run_id","run_date","txn_silver_reject_count"]
txn_silver_reject_count_df = spark.createDataFrame([data],cols)

txn_silver_reject_count_df.createOrReplaceTempView("txn_silver_reject_count_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING txn_silver_reject_count_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.txn_silver_reject_count = s.txn_silver_reject_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, txn_silver_reject_count) VALUES (s.run_id,s.run_date,s.txn_silver_reject_count)
""")

In [0]:
settlement_silver_count = event_log_df.filter(col("origin.flow_name")=="workspace.default.settlement_silver")\
.agg(sum(expr("details:flow_progress:metrics:num_output_rows").cast("int")).alias("settlement_silver_count")).collect()[0]["settlement_silver_count"]

if settlement_silver_count is None:
    settlement_silver_count =0

data =[run_id,run_date,settlement_silver_count]
cols=["run_id","run_date","settlement_silver_count"]
settlement_silver_count_df = spark.createDataFrame([data],cols)

settlement_silver_count_df.createOrReplaceTempView("settlement_silver_count_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING settlement_silver_count_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.settlement_silver_count = s.settlement_silver_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, settlement_silver_count) VALUES (s.run_id,s.run_date,s.settlement_silver_count)
""")

In [0]:
gold_count = event_log_df.filter(col("origin.flow_name")=="workspace.default.reconciled")\
.agg(sum(expr("details:flow_progress:metrics:num_output_rows").cast("int")).alias("gold_count")).collect()[0]["gold_count"]

if gold_count is None:
    gold_count =0

data =[run_id,run_date,gold_count]
cols=["run_id","run_date","gold_count"]
gold_count_df = spark.createDataFrame([data],cols)

gold_count_df.createOrReplaceTempView("gold_count_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING gold_count_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.gold_count = s.gold_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, gold_count) VALUES (s.run_id,s.run_date,s.gold_count)
""")

In [0]:
gold_discrep_unsettled_success_count = spark.read.table("workspace.default.reconciled").filter(col("reconciliation")=="DISCREPANCY UNSETTLED SUCCESS")\
.filter((col("ingested_time")<to_timestamp(lit(run_end))) & (col("ingested_time")>to_timestamp(lit(run_start)))).count()

if gold_discrep_unsettled_success_count is None:
    gold_discrep_unsettled_success_count =0

data =[run_id,run_date,gold_discrep_unsettled_success_count]
cols=["run_id","run_date","gold_discrep_unsettled_success_count"]
gold_discrep_unsettled_success_count_df = spark.createDataFrame([data],cols)

gold_discrep_unsettled_success_count_df.createOrReplaceTempView("gold_discrep_unsettled_success_count_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING gold_discrep_unsettled_success_count_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.gold_discrep_unsettled_success_count = s.gold_discrep_unsettled_success_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, gold_discrep_unsettled_success_count) VALUES (s.run_id,s.run_date,s.gold_discrep_unsettled_success_count)
""")

In [0]:
orphan_record_count = spark.read.table("workspace.default.reconciled").filter(col("reconciliation")=="ORPHAN RECORD")\
.filter((col("ingested_time")<to_timestamp(lit(run_end))) & (col("ingested_time")>to_timestamp(lit(run_start)))).count()

if orphan_record_count is None:
    orphan_record_count =0

data =[run_id,run_date,orphan_record_count]
cols=["run_id","run_date","orphan_record_count"]
orphan_record_count_df = spark.createDataFrame([data],cols)

orphan_record_count_df.createOrReplaceTempView("orphan_record_count_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING orphan_record_count_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.orphan_record_count = s.orphan_record_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, orphan_record_count) VALUES (s.run_id,s.run_date,s.orphan_record_count)
""")

In [0]:
gold_discrep_stale_pending_count = spark.read.table("workspace.default.reconciled").filter(col("reconciliation")=="DISCREPANCY STALE PENDING")\
.filter((col("ingested_time")<to_timestamp(lit(run_end))) & (col("ingested_time")>to_timestamp(lit(run_start)))).count()

if gold_discrep_stale_pending_count is None:
    gold_discrep_stale_pending_count =0

data =[run_id,run_date,gold_discrep_stale_pending_count]
cols=["run_id","run_date","gold_discrep_stale_pending_count"]
gold_discrep_stale_pending_count_df = spark.createDataFrame([data],cols)

gold_discrep_stale_pending_count_df.createOrReplaceTempView("gold_discrep_stale_pending_count_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING gold_discrep_stale_pending_count_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.gold_discrep_stale_pending_count = s.gold_discrep_stale_pending_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, gold_discrep_stale_pending_count) VALUES (s.run_id,s.run_date,s.gold_discrep_stale_pending_count)
""")

In [0]:
if gold_count>0:
    discrepancy_rate=(gold_discrep_unsettled_success_count+gold_discrep_stale_pending_count+orphan_record_count)/gold_count
else:
    discrepancy_rate=0

data =[run_id,run_date,discrepancy_rate]
cols=["run_id","run_date","discrepancy_rate"]
discrepancy_rate_df = spark.createDataFrame([data],cols)

discrepancy_rate_df.createOrReplaceTempView("discrepancy_rate_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING discrepancy_rate_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.discrepancy_rate = s.discrepancy_rate
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, discrepancy_rate) VALUES (s.run_id,s.run_date,s.discrepancy_rate)
""")

In [0]:
if txn_bronze_count>0:
    txn_rejection_rate=txn_silver_reject_count/txn_bronze_count
else:
    txn_rejection_rate=0


data =[run_id,run_date,txn_rejection_rate]
cols=["run_id","run_date","txn_rejection_rate"]
txn_rejection_rate_df = spark.createDataFrame([data],cols)

txn_rejection_rate_df.createOrReplaceTempView("txn_rejection_rate_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING txn_rejection_rate_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.txn_rejection_rate = s.txn_rejection_rate
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, txn_rejection_rate) VALUES (s.run_id,s.run_date,s.txn_rejection_rate)
""")

In [0]:
upi_transactions_bronze_range = event_log_df.filter(col("origin.flow_name")=="workspace.default.upi_transactions_bronze")

bronze_settlement_data_range = event_log_df.filter(col("origin.flow_name")=="workspace.default.bronze_settlement_data")

upi_transactions_bronze_range_boundaries = upi_transactions_bronze_range.agg(min("timestamp").alias("run_start"),max("timestamp").alias("run_end"))\
    .collect()[0]
bronze_settlement_data_boundaries = bronze_settlement_data_range.agg(min("timestamp").alias("run_start"),max("timestamp").alias("run_end"))\
    .collect()[0]

ranges_bronze = []

ranges_bronze.append(upi_transactions_bronze_range_boundaries["run_start"])
ranges_bronze.append(upi_transactions_bronze_range_boundaries["run_end"])

ranges_bronze.append(bronze_settlement_data_boundaries["run_start"])
ranges_bronze.append(bronze_settlement_data_boundaries["run_end"])

bronze_duration = (builtins.max(ranges_bronze)-builtins.min(ranges_bronze)).total_seconds()

data =[run_id,run_date,bronze_duration]
cols=["run_id","run_date","bronze_duration"]
bronze_duration_df = spark.createDataFrame([data],cols)

bronze_duration_df.createOrReplaceTempView("bronze_duration_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING bronze_duration_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.bronze_duration = s.bronze_duration
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, bronze_duration) VALUES (s.run_id,s.run_date,s.bronze_duration)
""")

In [0]:
ranges_silver = []

upi_transactions_bronze_inflated_range = event_log_df.filter(col("origin.flow_name")=="workspace.default.upi_transactions_bronze_inflated")
upi_transactions_bronze_inflated_range_boundaries = upi_transactions_bronze_inflated_range.agg(min("timestamp").alias("run_start"),max("timestamp").alias("run_end"))\
    .collect()[0]
ranges_silver.append(upi_transactions_bronze_inflated_range_boundaries["run_start"])
ranges_silver.append(upi_transactions_bronze_inflated_range_boundaries["run_end"])



upi_transactions_quarantine_range = event_log_df.filter(col("origin.flow_name")=="workspace.default.upi_transactions_quarantine")
upi_transactions_quarantine_range_boundaries = upi_transactions_quarantine_range.agg(min("timestamp").alias("run_start"),max("timestamp").alias("run_end"))\
    .collect()[0]
ranges_silver.append(upi_transactions_quarantine_range_boundaries["run_start"])
ranges_silver.append(upi_transactions_quarantine_range_boundaries["run_end"])


UPI_transactions_silver_range = event_log_df.filter(col("origin.flow_name")=="workspace.default.upi_transactions_silver")
upi_transactions_silver_range_boundaries = UPI_transactions_silver_range.agg(min("timestamp").alias("run_start"),max("timestamp").alias("run_end"))\
    .collect()[0]
ranges_silver.append(upi_transactions_silver_range_boundaries["run_start"])
ranges_silver.append(upi_transactions_silver_range_boundaries["run_end"])



settlement_silver_range = event_log_df.filter(col("origin.flow_name")=="workspace.default.settlement_silver")
settlement_silver_range_boundaries = settlement_silver_range.agg(min("timestamp").alias("run_start"),max("timestamp").alias("run_end"))\
    .collect()[0]
ranges_silver.append(settlement_silver_range_boundaries["run_start"])
ranges_silver.append(settlement_silver_range_boundaries["run_end"])



settlement_quarantine_range = event_log_df.filter(col("origin.flow_name")=="workspace.default.settlement_quarantine")   
settlement_quarantine_range_boundaries = settlement_quarantine_range.agg(min("timestamp").alias("run_start"),max("timestamp").alias("run_end"))\
    .collect()[0]
ranges_silver.append(settlement_quarantine_range_boundaries["run_start"])
ranges_silver.append(settlement_quarantine_range_boundaries["run_end"])



silver_duration = (builtins.max(ranges_silver)-builtins.min(ranges_silver)).total_seconds()

data =[run_id,run_date,silver_duration]
cols=["run_id","run_date","silver_duration"]
silver_duration_df = spark.createDataFrame([data],cols)

silver_duration_df.createOrReplaceTempView("silver_duration_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING silver_duration_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.silver_duration = s.silver_duration
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, silver_duration) VALUES (s.run_id,s.run_date,s.silver_duration)
""")

In [0]:
reconciled_range = event_log_df.filter(col("origin.flow_name")=="workspace.default.reconciled")

reconciled_range_boundaries = reconciled_range.agg(min("timestamp").alias("run_start"),max("timestamp").alias("run_end"))\
    .collect()[0]


ranges_gold = []

ranges_gold.append(reconciled_range_boundaries["run_start"])
ranges_gold.append(reconciled_range_boundaries["run_end"])

gold_duration = (builtins.max(ranges_gold)-builtins.min(ranges_gold)).total_seconds()

total_duration = gold_duration+bronze_duration+silver_duration

data =[run_id,run_date,gold_duration,total_duration]
cols=["run_id","run_date","gold_duration","total_duration"]
gold_and_total_duration_df = spark.createDataFrame([data],cols)

gold_and_total_duration_df.createOrReplaceTempView("gold_and_total_duration_temp")

spark.sql("""
    MERGE INTO workspace.default.upi_pipeline_metrics t
    USING gold_and_total_duration_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.gold_duration = s.gold_duration, t.total_duration = s.total_duration
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, gold_duration, total_duration) VALUES (s.run_id,s.run_date,s.gold_duration,s.total_duration)
""")